# SABR Model Calibration for Swaptions

## What Are We Building?

The SABR model is our toolkit for understanding how interest rate volatility changes across strikes and tenors. While Black's model assumes constant volatility, real markets show a **smile**: out-of-the-money options trade at different implied vols than at-the-money. SABR captures this by letting both the forward rate and volatility itself move randomly, with a controllable correlation between them.

This notebook walks through the full calibration pipeline: from market data, through the optimization, to pricing and risk metrics. By the end, you'll have a working swaption pricer grounded in SABR and a deep intuition for what each parameter does.

---

## The SABR Model -- From First Principles

**SABR** stands for **Stochastic Alpha Beta Rho**. It models how the forward rate $F$ and its volatility $\alpha$ evolve over time. The key insight is that volatility itself is random -- it moves around just like the forward rate does. This is what creates the "smile" pattern we see in real option markets.

### The Equations

$$dF = \alpha \, F^{\beta} \, dW_1$$
$$d\alpha = \nu \, \alpha \, dW_2$$

with correlation $\text{corr}(dW_1, dW_2) = \rho$.

### The Parameters

| Parameter | Name | Typical Range | What It Controls |
|-----------|------|---------------|------------------|
| $\alpha$ | Initial volatility | 0.01 -- 0.10 | Overall level of the vol smile (ATM vol) |
| $\beta$ | CEV exponent | 0.0 -- 1.0 | Backbone: how vol depends on the forward rate level |
| $\nu$ | Vol of vol | 0.3 -- 2.0 | Width of the smile -- higher $\nu$ means wider wings |
| $\rho$ | Correlation | -0.7 to -0.3 | Skew direction -- negative $\rho$ tilts vol higher for low strikes |

**Intuition:**
- $\beta$ controls how much the forward rate's own level affects its volatility. $\beta = 1$ means lognormal dynamics (percentage moves); $\beta = 0$ means normal dynamics (absolute moves). In practice, $\beta$ is often fixed at 0.5 or calibrated.
- $\nu$ (vol of vol) is literally how much the volatility itself bounces around. Higher $\nu$ = wider smile.
- $\rho$ (correlation) determines skew. Negative $\rho$ means when the forward rate drops, vol tends to rise -- this is the classic "leverage effect" that makes put protection expensive.

### What We'll Build

1. **Load market data** -- swaption implied volatilities across strikes and tenors
2. **Implement Hagan's approximation** -- the closed-form formula that maps SABR parameters to implied vol
3. **Calibrate** -- fit SABR parameters to market data via optimization
4. **Visualize** -- plot the fit, the parameter surface, and Greeks
5. **Price** -- use calibrated SABR to price swaptions and compute risk sensitivities

### Greeks We'll Track

- **Vega**: How much the swaption price changes when implied vol shifts by 1bp
- **Gamma**: How much delta (rate sensitivity) changes as the forward rate moves -- measures convexity
- **Vanna**: The cross-derivative -- how vega changes as the forward rate moves. This matters for managing the interaction between rate risk and vol risk.

---

## Market Data & Setup

Before we can calibrate anything, we need market data. A swaption volatility surface is a grid of implied volatilities organized by two dimensions:

- **Option expiry** (how long until the swaption expires -- e.g., 1Y, 5Y, 10Y)
- **Strike** (the fixed rate at which the holder can enter the swap)

For each combination of expiry and underlying swap tenor, dealers quote implied volatilities at several strikes around the at-the-money (ATM) forward rate. These quotes can be in **lognormal** (Black) or **normal** (Bachelier) convention -- we'll work with lognormal (Black) vols here, which is what SABR's Hagan formula naturally produces.

In a production setting, you'd pull this data from Bloomberg's VCUB function. Here we'll use realistic synthetic data that mimics the shape of a real swaption vol surface.

The cell below imports our libraries and creates a sample volatility dataset. We'll structure it as a dictionary keyed by `(expiry, tenor)` pairs, each containing arrays of strikes and corresponding implied vols.

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0',
    'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0',
    'grid.color': '#2a2a4a',
    'legend.facecolor': '#16213e',
    'legend.edgecolor': '#e0e0e0',
    'figure.figsize': (12, 6),
    'font.size': 11
})

def generate_sample_vol_surface():
    expiries = [1, 2, 5, 10]
    tenors = [2, 5, 10]
    strike_offsets_bp = np.array([-200, -100, -50, -25, 0, 25, 50, 100, 200])

    surface = {}

    for expiry in expiries:
        for tenor in tenors:
            atm_forward = 0.03 + 0.002 * tenor + 0.001 * expiry
            strikes = atm_forward + strike_offsets_bp / 10000

            atm_vol = 0.20 + 0.05 * np.exp(-0.1 * expiry) + 0.01 * tenor
            skew = -0.0015 * (strikes - atm_forward) / atm_forward
            curvature = 0.8 * ((strikes - atm_forward) / atm_forward) ** 2
            noise = np.random.RandomState(42 + expiry * 10 + tenor).normal(0, 0.002, len(strikes))

            implied_vols = atm_vol + skew + curvature + noise
            implied_vols = np.maximum(implied_vols, 0.01)

            surface[(expiry, tenor)] = {
                'strikes': strikes,
                'atm_forward': atm_forward,
                'implied_vols': implied_vols,
                'strike_offsets_bp': strike_offsets_bp
            }

    return surface, expiries, tenors

surface, expiries, tenors = generate_sample_vol_surface()

example = surface[(5, 10)]
print("Example: 5Y expiry into 10Y swap")
print(f"  ATM Forward: {example['atm_forward']:.4f} ({example['atm_forward']*100:.2f}%)")
print(f"  Strikes:     {np.round(example['strikes'] * 100, 2)} (%)")
print(f"  Impl Vols:   {np.round(example['implied_vols'] * 100, 2)} (%)")

Let's visualize one volatility smile to see the shape we're trying to fit. The characteristic U-shape (or skewed U) is exactly what SABR is designed to capture.

In [ ]:
example = surface[(5, 10)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(example['strike_offsets_bp'], example['implied_vols'] * 100,
        'o-', color='#00d2ff', markersize=8, linewidth=2, label='Market vols')
ax.axvline(0, color='#ff6b6b', linestyle='--', alpha=0.6, label='ATM')
ax.set_xlabel('Strike Offset from ATM (bp)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('Swaption Vol Smile -- 5Y Expiry into 10Y Swap', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## The Hagan Approximation

The full SABR stochastic differential equations don't have a closed-form solution for implied volatility. Luckily, Hagan, Kumar, Lesniewski, and Woodward (2002) derived an asymptotic approximation that maps the four SABR parameters directly to a Black implied volatility for any given strike and forward.

The Hagan approximation lets us **compute implied vol instantly from SABR parameters** without **running Monte Carlo simulations or solving PDEs**. It's fast, accurate, and differentiable -- perfect for optimization.

### How the Formula Works

The Hagan formula takes as input a strike $K$, forward rate $F$, the SABR parameters $(\alpha, \beta, \nu, \rho)$, and time to expiry $T$. It returns the Black implied volatility $\sigma_B(K)$. The formula has three main pieces:

1. **A base term** that depends on $\alpha$ and $\beta$ -- this sets the overall vol level
2. **A z-transform** involving $\nu$ and $\rho$ -- this generates the smile shape
3. **Correction terms** of order $T$ -- small adjustments that improve accuracy

The vol smile emerges because $\beta$ and $\rho$ create asymmetry (skew) in how vol responds to strikes above vs below ATM, while $\nu$ controls the overall curvature (width of the smile wings).

The cell below implements this formula. We handle the special case where $K = F$ (the ATM case) separately, since the general formula has a $\log(F/K)$ term that becomes zero.

In [ ]:
def hagan_implied_vol(strike, forward, expiry, alpha, beta, nu, rho):
    K = np.asarray(strike, dtype=float)
    F = float(forward)
    T = float(expiry)
    eps = 1e-12

    FK = F * K
    FK_mid = np.sqrt(FK)
    log_FK = np.log(F / np.maximum(K, eps))

    one_minus_beta = 1.0 - beta
    FK_pow = FK_mid ** one_minus_beta

    atm_mask = np.abs(log_FK) < 1e-7

    # z-transform for smile shape
    z = (nu / alpha) * FK_pow * log_FK
    x = np.log((np.sqrt(1 - 2 * rho * z + z**2) + z - rho) / (1 - rho + eps) + eps)
    zx_ratio = np.where(atm_mask, 1.0, z / (x + eps))

    # denominator: (1-beta) expansion of log(F/K)
    denom_term = (one_minus_beta * log_FK)**2
    denom = FK_pow * (1 + denom_term / 24 + denom_term**2 / 1920)

    # numerator: order-T correction
    term1 = (one_minus_beta * alpha)**2 / (24 * FK_mid**(2 * one_minus_beta))
    term2 = 0.25 * rho * beta * nu * alpha / FK_pow
    term3 = (2 - 3 * rho**2) * nu**2 / 24
    numerator = 1 + (term1 + term2 + term3) * T

    sigma = (alpha / denom) * zx_ratio * numerator
    return np.maximum(sigma, eps)


# sanity check: ATM vol for known parameters
test_vol = hagan_implied_vol(
    strike=0.04, forward=0.04, expiry=5.0,
    alpha=0.035, beta=0.5, nu=0.5, rho=-0.3
)
print(f"SABR ATM implied vol: {test_vol*100:.2f}%")

# show the smile across a range of strikes
test_strikes = np.linspace(0.01, 0.07, 100)
test_vols = hagan_implied_vol(test_strikes, 0.04, 5.0, 0.035, 0.5, 0.5, -0.3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test_strikes * 100, test_vols * 100, color='#00d2ff', linewidth=2)
ax.axvline(4.0, color='#ff6b6b', linestyle='--', alpha=0.6, label='ATM = 4%')
ax.set_xlabel('Strike (%)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('SABR Smile from Hagan Formula', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Checkpoint:** Notice the asymmetry in the smile -- the left wing (low strikes) has higher vol than the right wing. That's the skew, driven by negative $\rho$. Try changing $\rho$ to 0 in the test above -- the smile becomes symmetric. Try increasing $\nu$ -- the wings get wider.

---

## Calibration: The Optimization Problem

Now we have a formula that turns SABR parameters into implied vols. Calibration is the reverse: given market vols, find the parameters that reproduce them as closely as possible.

Calibration is an **optimization problem** where we minimize the **squared difference between market and model vols** subject to **parameter bounds**:

$$\min_{\alpha, \nu, \rho} \sum_{i} \left( \sigma^{\text{market}}_i - \sigma^{\text{SABR}}_i(\alpha, \beta, \nu, \rho; K_i, F, T) \right)^2$$

subject to $\alpha > 0$, $\nu > 0$, and $-1 < \rho < 1$.

A few practical notes:
- We typically **fix $\beta$** (e.g., at 0.5) and calibrate $\alpha$, $\nu$, $\rho$. Trying to calibrate all four simultaneously leads to over-fitting and parameter instability.
- The optimizer needs good initial guesses. A reasonable starting point: $\alpha \approx$ ATM vol $\times F^{1-\beta}$, $\nu \approx 0.5$, $\rho \approx -0.3$.
- We use `scipy.optimize.minimize` with the L-BFGS-B method, which handles box constraints naturally.

The cell below defines the objective function and a calibration wrapper that handles bounds and initial guesses.

In [ ]:
def sabr_objective(params, strikes, forward, expiry, market_vols, beta):
    alpha, nu, rho = params
    if alpha <= 0 or nu <= 0 or abs(rho) >= 1:
        return 1e10
    model_vols = hagan_implied_vol(strikes, forward, expiry, alpha, beta, nu, rho)
    residuals = model_vols - market_vols
    return np.sum(residuals ** 2)


def calibrate_sabr(strikes, forward, expiry, market_vols, beta=0.5):
    atm_vol_guess = np.interp(forward, strikes, market_vols)
    alpha0 = atm_vol_guess * forward ** (1 - beta)
    x0 = [alpha0, 0.5, -0.3]

    bounds = [
        (1e-6, 1.0),      # alpha
        (1e-4, 5.0),      # nu
        (-0.999, 0.999)   # rho
    ]

    result = minimize(
        sabr_objective, x0,
        args=(strikes, forward, expiry, market_vols, beta),
        method='L-BFGS-B', bounds=bounds,
        options={'maxiter': 1000, 'ftol': 1e-14}
    )

    alpha_cal, nu_cal, rho_cal = result.x
    model_vols = hagan_implied_vol(strikes, forward, expiry, alpha_cal, beta, nu_cal, rho_cal)
    rmse = np.sqrt(np.mean((model_vols - market_vols) ** 2))

    return {
        'alpha': alpha_cal, 'beta': beta, 'nu': nu_cal, 'rho': rho_cal,
        'rmse': rmse, 'model_vols': model_vols, 'converged': result.success
    }

print("Calibration functions ready.")

---

## Calibrating a Single Volatility Smile

Let's start with one slice of the surface: the **5Y expiry into 10Y swap**. We'll run the calibrator and see how well SABR reproduces the market smile.

This is the moment of truth for the model. A good fit means the Hagan formula, with our three free parameters, can capture the shape of the market's smile. We'll overlay the model curve on the market data points and check the RMSE (root mean square error) -- anything under 50bp is a solid fit for most practical purposes.

In [ ]:
data = surface[(5, 10)]
result = calibrate_sabr(
    data['strikes'], data['atm_forward'], 5.0, data['implied_vols'], beta=0.5
)

print("Calibrated SABR Parameters (5Y expiry, 10Y swap):")
print(f"  alpha = {result['alpha']:.6f}")
print(f"  beta  = {result['beta']:.1f}  (fixed)")
print(f"  nu    = {result['nu']:.4f}")
print(f"  rho   = {result['rho']:.4f}")
print(f"  RMSE  = {result['rmse']*10000:.2f} bp")
print(f"  Converged = {result['converged']}")

fine_strikes = np.linspace(data['strikes'][0], data['strikes'][-1], 200)
fine_vols = hagan_implied_vol(
    fine_strikes, data['atm_forward'], 5.0,
    result['alpha'], result['beta'], result['nu'], result['rho']
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(data['strike_offsets_bp'], data['implied_vols'] * 100,
        'o', color='#00d2ff', markersize=9, label='Market', zorder=5)
ax.plot((fine_strikes - data['atm_forward']) * 10000, fine_vols * 100,
        '-', color='#ff6b6b', linewidth=2.5, label='SABR fit')
ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
ax.set_xlabel('Strike Offset from ATM (bp)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('SABR Calibration -- 5Y into 10Y', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Checkpoint:** The SABR curve should pass close to every market point. If the wings deviate, it's usually a sign that $\nu$ needs more freedom or that $\beta$ should be adjusted. Notice how negative $\rho$ produces the downward-sloping left side of the smile -- that's the skew that makes low-strike (receiver) swaptions more expensive in vol terms.

---

## Building the Full Volatility Surface

Now we scale up: calibrate SABR parameters for every (expiry, tenor) pair in the surface. This gives us a view of how $\alpha$, $\nu$, and $\rho$ vary across the term structure.

Understanding the parameter surface is valuable for trading:
- If $\nu$ (vol of vol) is highest for short-dated options, the market prices more smile convexity in the near term.
- If $\rho$ becomes more negative for longer tenors, the skew is steeper there -- perhaps reflecting term premium or structural demand for receiver protection.

In [ ]:
calibration_results = {}

print(f"{'Expiry':>6} {'Tenor':>5} {'alpha':>8} {'nu':>8} {'rho':>8} {'RMSE(bp)':>9} {'OK':>4}")
print("-" * 52)

for expiry in expiries:
    for tenor in tenors:
        data = surface[(expiry, tenor)]
        res = calibrate_sabr(
            data['strikes'], data['atm_forward'], float(expiry), data['implied_vols'], beta=0.5
        )
        calibration_results[(expiry, tenor)] = res

        status = 'Y' if res['converged'] else 'N'
        print(f"{expiry:>4}Y {tenor:>3}Y {res['alpha']:>8.5f} {res['nu']:>8.4f} "
              f"{res['rho']:>8.4f} {res['rmse']*10000:>8.2f} {status:>4}")

Let's visualize how each parameter behaves across the surface -- this is the **term structure of SABR parameters**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
param_names = ['alpha', 'nu', 'rho']
param_labels = ['alpha (ATM Vol Level)', 'nu (Vol of Vol)', 'rho (Correlation / Skew)']

for ax, pname, plabel in zip(axes, param_names, param_labels):
    for tenor in tenors:
        values = [calibration_results[(exp, tenor)][pname] for exp in expiries]
        ax.plot(expiries, values, 'o-', label=f'{tenor}Y tenor', linewidth=2, markersize=7)
    ax.set_xlabel('Option Expiry (years)')
    ax.set_ylabel(pname)
    ax.set_title(plabel, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Now let's see all the calibrated smiles together to confirm the fits look good across the entire surface.

In [ ]:
fig, axes = plt.subplots(len(expiries), len(tenors), figsize=(18, 4 * len(expiries)),
                         sharex=True)

for i, expiry in enumerate(expiries):
    for j, tenor in enumerate(tenors):
        ax = axes[i, j]
        data = surface[(expiry, tenor)]
        res = calibration_results[(expiry, tenor)]

        fine_strikes = np.linspace(data['strikes'][0], data['strikes'][-1], 200)
        fine_vols = hagan_implied_vol(
            fine_strikes, data['atm_forward'], float(expiry),
            res['alpha'], res['beta'], res['nu'], res['rho']
        )

        ax.plot(data['strike_offsets_bp'], data['implied_vols'] * 100,
                'o', color='#00d2ff', markersize=5)
        ax.plot((fine_strikes - data['atm_forward']) * 10000, fine_vols * 100,
                '-', color='#ff6b6b', linewidth=1.5)
        ax.set_title(f'{expiry}Yx{tenor}Y  (RMSE={res["rmse"]*10000:.1f}bp)',
                     fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.2)

        if i == len(expiries) - 1:
            ax.set_xlabel('Strike Offset (bp)')
        if j == 0:
            ax.set_ylabel('Impl Vol (%)')

plt.suptitle('SABR Calibration Across the Full Surface', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## Greeks in the SABR Framework

Greeks measure how a swaption's price changes when market inputs move. They're the bread and butter of risk management. In the SABR context, we compute Greeks by combining two pieces:

1. **Black's formula** -- gives us the swaption price from an implied vol
2. **Hagan's SABR formula** -- gives us the implied vol from SABR parameters

By chaining these together, we can compute how the price responds to shifts in the forward rate, volatility, or both.

### The Greeks We'll Compute

- **Vega**: How much the swaption price changes per 1bp shift in implied volatility. This is your primary vol exposure.
- **Gamma**: How much delta changes as the forward rate moves. High gamma means your hedge needs frequent rebalancing.
- **Vanna**: The cross-derivative $\partial^2 P / \partial F \partial \sigma$. It tells you how your vega exposure changes as the forward moves -- critical for understanding the interaction between rate and vol risk.

These Greeks matter for trading because they determine how frequently and aggressively a desk needs to rebalance hedges, and how exposed a portfolio is to joint moves in rates and vol.

We'll compute them using **finite differences**: bump the input, recompute the price, measure the change.

In [ ]:
from scipy.stats import norm

def black_price(forward, strike, expiry, vol, is_payer=True):
    if vol <= 0 or expiry <= 0:
        return max(forward - strike, 0) if is_payer else max(strike - forward, 0)
    d1 = (np.log(forward / strike) + 0.5 * vol**2 * expiry) / (vol * np.sqrt(expiry))
    d2 = d1 - vol * np.sqrt(expiry)
    if is_payer:
        return forward * norm.cdf(d1) - strike * norm.cdf(d2)
    else:
        return strike * norm.cdf(-d2) - forward * norm.cdf(-d1)


def sabr_price(forward, strike, expiry, alpha, beta, nu, rho, is_payer=True):
    vol = hagan_implied_vol(strike, forward, expiry, alpha, beta, nu, rho)
    return black_price(forward, strike, expiry, float(vol), is_payer)


def compute_greeks(forward, strike, expiry, alpha, beta, nu, rho, is_payer=True):
    dF = forward * 1e-4
    dVol = 1e-4

    price_base = sabr_price(forward, strike, expiry, alpha, beta, nu, rho, is_payer)

    # vega: bump alpha
    p_up_v = sabr_price(forward, strike, expiry, alpha + dVol, beta, nu, rho, is_payer)
    p_dn_v = sabr_price(forward, strike, expiry, alpha - dVol, beta, nu, rho, is_payer)
    vega = (p_up_v - p_dn_v) / (2 * dVol) * 0.0001

    # gamma: second derivative wrt forward
    p_up_F = sabr_price(forward + dF, strike, expiry, alpha, beta, nu, rho, is_payer)
    p_dn_F = sabr_price(forward - dF, strike, expiry, alpha, beta, nu, rho, is_payer)
    gamma = (p_up_F - 2 * price_base + p_dn_F) / (dF ** 2)

    # vanna: cross-derivative d(vega)/d(forward)
    vega_up = (sabr_price(forward + dF, strike, expiry, alpha + dVol, beta, nu, rho, is_payer)
             - sabr_price(forward + dF, strike, expiry, alpha - dVol, beta, nu, rho, is_payer)) / (2 * dVol)
    vega_dn = (sabr_price(forward - dF, strike, expiry, alpha + dVol, beta, nu, rho, is_payer)
             - sabr_price(forward - dF, strike, expiry, alpha - dVol, beta, nu, rho, is_payer)) / (2 * dVol)
    vanna = (vega_up - vega_dn) / (2 * dF)

    return {'price': price_base, 'vega': vega, 'gamma': gamma, 'vanna': vanna}


# compute Greeks across strikes for 5Y10Y
data = surface[(5, 10)]
res = calibration_results[(5, 10)]

greeks_by_strike = []
for K in data['strikes']:
    g = compute_greeks(data['atm_forward'], K, 5.0,
                       res['alpha'], res['beta'], res['nu'], res['rho'])
    greeks_by_strike.append(g)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
greek_names = ['price', 'vega', 'gamma', 'vanna']
greek_labels = ['Swaption Price', 'Vega (per 1bp)', 'Gamma', 'Vanna']
greek_colors = ['#00d2ff', '#ff6b6b', '#ffd93d', '#a855f7']

for ax, gn, gl, gc in zip(axes.flat, greek_names, greek_labels, greek_colors):
    vals = [g[gn] for g in greeks_by_strike]
    ax.plot(data['strike_offsets_bp'], vals, 'o-', color=gc, linewidth=2, markersize=7)
    ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
    ax.set_xlabel('Strike Offset (bp)')
    ax.set_ylabel(gl)
    ax.set_title(gl, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('SABR Greeks -- 5Y into 10Y Payer Swaption', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Checkpoint:** A few things to notice:

- **Vega** peaks near ATM and falls off in the wings -- ATM options are most sensitive to vol shifts.
- **Gamma** also peaks near ATM -- this is where delta changes fastest as the forward moves.
- **Vanna** changes sign around ATM -- this cross-effect tells you that for low strikes, your vega exposure increases as the forward drops (and vice versa). This is why managing a book of swaptions with skew exposure requires monitoring vanna carefully.

### Greeks Across the Surface

Let's compute vega for every (expiry, tenor) combination at ATM to see where vol sensitivity concentrates.

In [ ]:
vega_grid = np.zeros((len(expiries), len(tenors)))

for i, expiry in enumerate(expiries):
    for j, tenor in enumerate(tenors):
        data = surface[(expiry, tenor)]
        res = calibration_results[(expiry, tenor)]
        g = compute_greeks(data['atm_forward'], data['atm_forward'], float(expiry),
                           res['alpha'], res['beta'], res['nu'], res['rho'])
        vega_grid[i, j] = g['vega'] * 10000

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(vega_grid, cmap='magma', aspect='auto', origin='lower')
ax.set_xticks(range(len(tenors)))
ax.set_xticklabels([f'{t}Y' for t in tenors])
ax.set_yticks(range(len(expiries)))
ax.set_yticklabels([f'{e}Y' for e in expiries])
ax.set_xlabel('Swap Tenor')
ax.set_ylabel('Option Expiry')
ax.set_title('ATM Vega Heatmap (x10000)', fontsize=14, fontweight='bold')

for i in range(len(expiries)):
    for j in range(len(tenors)):
        ax.text(j, i, f'{vega_grid[i,j]:.2f}', ha='center', va='center',
                color='white' if vega_grid[i,j] < vega_grid.mean() else 'black', fontsize=12)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---

## Putting It All Together: Pricing a Swaption

Let's use our calibrated SABR parameters to price a specific swaption and break down the risk.

Suppose a trader wants to know the value and risk of a **5Y payer swaption on a 10Y swap** struck 50bp out of the money. Using our calibrated model, we can compute the price (as a fraction of notional) and all the relevant Greeks.

This is exactly how a trading desk would use SABR: calibrate to the market once (or a few times per day), then reprice the entire book using the calibrated parameters -- no need to re-optimize for each trade.

In [ ]:
data = surface[(5, 10)]
res = calibration_results[(5, 10)]

strike_otm = data['atm_forward'] + 0.005  # 50bp OTM

sabr_vol = hagan_implied_vol(
    strike_otm, data['atm_forward'], 5.0,
    res['alpha'], res['beta'], res['nu'], res['rho']
)

greeks = compute_greeks(
    data['atm_forward'], strike_otm, 5.0,
    res['alpha'], res['beta'], res['nu'], res['rho'],
    is_payer=True
)

notional = 100_000_000
annuity = sum(1 / (1 + data['atm_forward'])**i for i in range(1, 11))

print("=" * 55)
print("  SWAPTION PRICING -- SABR MODEL")
print("=" * 55)
print(f"  Instrument:    5Y Payer Swaption on 10Y Swap")
print(f"  ATM Forward:   {data['atm_forward']*100:.3f}%")
print(f"  Strike:        {strike_otm*100:.3f}% (+50bp OTM)")
print(f"  SABR Vol:      {float(sabr_vol)*100:.2f}%")
print(f"  Notional:      ${notional:,.0f}")
print("-" * 55)
print(f"  Price (norm):  {greeks['price']:.6f}")
print(f"  Price ($):     ${greeks['price'] * annuity * notional:,.0f}")
print("-" * 55)
print(f"  Vega (1bp):    ${greeks['vega'] * annuity * notional:,.0f}")
print(f"  Gamma:         {greeks['gamma']:.4f}")
print(f"  Vanna:         {greeks['vanna']:.6f}")
print("=" * 55)

### Sensitivity Analysis: What Happens When Parameters Shift?

Let's see how the swaption price responds to changes in each SABR parameter. This builds intuition for what each parameter "does" to the price.

In [ ]:
data = surface[(5, 10)]
res = calibration_results[(5, 10)]
strike_atm = data['atm_forward']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# alpha sensitivity
alphas = np.linspace(res['alpha'] * 0.5, res['alpha'] * 1.5, 50)
prices_a = [sabr_price(data['atm_forward'], strike_atm, 5.0, a, res['beta'], res['nu'], res['rho'])
             for a in alphas]
axes[0].plot(alphas, prices_a, color='#00d2ff', linewidth=2.5)
axes[0].axvline(res['alpha'], color='#ff6b6b', linestyle='--', alpha=0.6,
                label=f"Calibrated={res['alpha']:.4f}")
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('ATM Price')
axes[0].set_title('Price vs alpha (Vol Level)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# nu sensitivity
nus = np.linspace(0.1, 2.0, 50)
prices_n = [sabr_price(data['atm_forward'], strike_atm, 5.0, res['alpha'], res['beta'], n, res['rho'])
             for n in nus]
axes[1].plot(nus, prices_n, color='#ffd93d', linewidth=2.5)
axes[1].axvline(res['nu'], color='#ff6b6b', linestyle='--', alpha=0.6,
                label=f"Calibrated={res['nu']:.4f}")
axes[1].set_xlabel('nu')
axes[1].set_ylabel('ATM Price')
axes[1].set_title('Price vs nu (Vol of Vol)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# rho sensitivity
rhos = np.linspace(-0.95, 0.95, 50)
prices_r = [sabr_price(data['atm_forward'], strike_atm, 5.0, res['alpha'], res['beta'], res['nu'], r)
             for r in rhos]
axes[2].plot(rhos, prices_r, color='#a855f7', linewidth=2.5)
axes[2].axvline(res['rho'], color='#ff6b6b', linestyle='--', alpha=0.6,
                label=f"Calibrated={res['rho']:.4f}")
axes[2].set_xlabel('rho')
axes[2].set_ylabel('ATM Price')
axes[2].set_title('Price vs rho (Correlation)', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('ATM Swaption Price Sensitivity to SABR Parameters', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Real Market Data & 3D Implied Volatility Surface

Up to this point we have been working with synthetic data. This section attempts to pull **real market data** and then plots a 3D implied volatility surface for the **10Y swap tenor** -- showing how the vol smile evolves across option expiries.

### Data Source Cascade

We try three sources in order:

1. **gs_quant** (Goldman Sachs Marquee) -- the gold standard for free swaption vol data, but requires a Marquee account and API credentials.
2. **FRED** (Federal Reserve Economic Data) -- we pull 10Y USD swap rates, compute annualized historical volatility as an ATM proxy, then build a synthetic smile around each expiry point using typical skew and curvature parameters.
3. **Synthetic fallback** -- if neither source works, we reuse the existing synthetic surface and print a clear message.

The 3D surface fixes the swap tenor at 10Y and sweeps across strikes (x-axis, in %) and option expiries (y-axis, in years), with implied volatility on the z-axis. This is the view a rates trader would use to spot relative value across the expiry/strike grid.

In [ ]:
import datetime as dt
from scipy.interpolate import RegularGridInterpolator

# ---- helpers ----

def build_smile_from_atm(atm_forward, atm_vol, strike_offsets_bp,
                         skew_strength=-0.3, curvature_strength=0.8):
    strikes = atm_forward + strike_offsets_bp / 10000
    moneyness = (strikes - atm_forward) / atm_forward
    skew = skew_strength * moneyness * atm_vol
    curvature = curvature_strength * moneyness**2 * atm_vol
    implied_vols = np.maximum(atm_vol + skew + curvature, 0.005)
    return strikes, implied_vols


def attempt_gs_quant(target_expiries, target_tenor=10):
    """Try pulling swaption vols from Goldman Sachs Marquee."""
    try:
        from gs_quant.session import GsSession, Environment
        from gs_quant.timeseries import implied_volatility
        print("[gs_quant] Attempting Marquee connection...")
        GsSession.use(Environment.PROD, client_id=None, client_secret=None)
        offsets = np.array([-200, -100, -50, -25, 0, 25, 50, 100, 200])
        gs_surface = {}
        for exp in target_expiries:
            atm_fwd = 0.03 + 0.002 * target_tenor + 0.001 * exp
            series = implied_volatility(f"{exp}y{target_tenor}y", tenor=f"{target_tenor}y")
            atm_vol = float(series.iloc[-1]) / 100
            strikes, vols = build_smile_from_atm(atm_fwd, atm_vol, offsets)
            gs_surface[(exp, target_tenor)] = {
                'strikes': strikes, 'atm_forward': atm_fwd,
                'implied_vols': vols, 'strike_offsets_bp': offsets
            }
        print("[gs_quant] Success.")
        return gs_surface, True
    except Exception as e:
        print(f"[gs_quant] Not available: {type(e).__name__}: {e}")
        return {}, False


def attempt_fred(target_expiries, target_tenor=10):
    """Pull swap/Treasury rates from FRED, compute hist vol, build smiles."""
    try:
        import pandas_datareader.data as pdr
        end_date = dt.date.today()
        start_date = end_date - dt.timedelta(days=500)
        offsets = np.array([-200, -100, -50, -25, 0, 25, 50, 100, 200])
        rate_data = None
        for sid, lbl in [('DSWP10', '10Y swap rate'), ('DGS10', '10Y Treasury')]:
            try:
                print(f"[FRED] Fetching {lbl} ({sid})...")
                rate_data = pdr.DataReader(sid, 'fred', start_date, end_date).dropna()
                if len(rate_data) > 50:
                    print(f"[FRED] Got {len(rate_data)} observations.")
                    break
                rate_data = None
            except Exception:
                rate_data = None
        if rate_data is None or len(rate_data) < 50:
            raise ValueError("Not enough data from FRED")
        rates = rate_data.iloc[:, 0].values / 100
        current_rate = rates[-1]
        log_rets = np.diff(np.log(rates[rates > 0]))
        hist_vol = np.std(log_rets) * np.sqrt(252)
        print(f"[FRED] Rate: {current_rate*100:.2f}%  Hist vol: {hist_vol*100:.1f}%")
        fred_surface = {}
        for exp in target_expiries:
            atm_fwd = current_rate + 0.001 * exp
            atm_vol = hist_vol * (1 + 0.1 * np.exp(-0.15 * exp))
            strikes, vols = build_smile_from_atm(atm_fwd, atm_vol, offsets)
            fred_surface[(exp, target_tenor)] = {
                'strikes': strikes, 'atm_forward': atm_fwd,
                'implied_vols': vols, 'strike_offsets_bp': offsets
            }
        print("[FRED] Surface built.")
        return fred_surface, True
    except Exception as e:
        print(f"[FRED] Not available: {type(e).__name__}: {e}")
        return {}, False


# ---- run the cascade ----

target_expiries = [1, 2, 5, 10]
target_tenor = 10
data_source = "synthetic"

real_surface, success = attempt_gs_quant(target_expiries, target_tenor)
if success: data_source = "gs_quant"

if not success:
    real_surface, success = attempt_fred(target_expiries, target_tenor)
    if success: data_source = "fred"

if not success:
    print("\n** Could not fetch real market data -- using synthetic surface. **")
    print("   Install pandas-datareader or gs-quant for real data.\n")
    real_surface = {k: v for k, v in surface.items() if k[1] == target_tenor}

print(f"\nData source: {data_source}")
print(f"Slices: {sorted(real_surface.keys())}")

### 3D Implied Volatility Surface

Now we build the 3D surface. We take the vol smiles for each expiry (all at 10Y swap tenor), calibrate SABR to each one, stack them into a grid, interpolate to get a smooth mesh, and plot it.

The x-axis is strike (in %), the y-axis is option expiry (in years), and the z-axis is implied volatility (in %). The colormap encodes vol level -- warmer colors mean higher vol. You should see the smile shape repeated at each expiry, with the overall level shifting as you move along the expiry axis.

In [ ]:
# ---- calibrate each slice and build the vol grid ----

surface_expiries = sorted([k[0] for k in real_surface.keys()])
all_strike_sets = [real_surface[(e, target_tenor)]['strikes'] for e in surface_expiries]

global_k_min = max(s.min() for s in all_strike_sets)
global_k_max = min(s.max() for s in all_strike_sets)
fine_strikes = np.linspace(global_k_min, global_k_max, 80)

vol_matrix = np.zeros((len(surface_expiries), len(fine_strikes)))

for i, exp in enumerate(surface_expiries):
    sl = real_surface[(exp, target_tenor)]
    cal = calibrate_sabr(sl['strikes'], sl['atm_forward'],
                         float(exp), sl['implied_vols'], beta=0.5)
    model_vols = hagan_implied_vol(fine_strikes, sl['atm_forward'], float(exp),
                                   cal['alpha'], cal['beta'], cal['nu'], cal['rho'])
    vol_matrix[i, :] = model_vols * 100
    print(f"  {exp:>2}Y:  a={cal['alpha']:.5f}  nu={cal['nu']:.4f}  rho={cal['rho']:.4f}  RMSE={cal['rmse']*10000:.1f}bp")


# ---- interpolate to a smooth expiry axis ----

fine_expiries = np.linspace(min(surface_expiries), max(surface_expiries), 60)
interp = RegularGridInterpolator(
    (np.array(surface_expiries, dtype=float), fine_strikes),
    vol_matrix, method='linear', bounds_error=False, fill_value=None
)

strike_mesh, expiry_mesh = np.meshgrid(fine_strikes, fine_expiries)
pts = np.column_stack([expiry_mesh.ravel(), strike_mesh.ravel()])
vol_smooth = interp(pts).reshape(expiry_mesh.shape)


# ---- 3D surface plot ----

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    strike_mesh * 100, expiry_mesh, vol_smooth,
    cmap='plasma', edgecolor='none', alpha=0.92
)

# overlay actual data points as cyan dots
for i, exp in enumerate(surface_expiries):
    sl = real_surface[(exp, target_tenor)]
    ax.scatter(sl['strikes'] * 100,
               np.full_like(sl['strikes'], exp),
               sl['implied_vols'] * 100,
               color='#00d2ff', s=25, zorder=10,
               edgecolors='white', linewidth=0.5)

ax.set_xlabel('Strike (%)', fontsize=12, labelpad=10)
ax.set_ylabel('Option Expiry (years)', fontsize=12, labelpad=10)
ax.set_zlabel('Implied Vol (%)', fontsize=12, labelpad=10)

source_labels = {'gs_quant': 'GS Marquee', 'fred': 'FRED + Synthetic Smile', 'synthetic': 'Synthetic'}
ax.set_title(
    f'Swaption Implied Vol Surface -- 10Y Swap Tenor\n'
    f'Data: {source_labels[data_source]}',
    fontsize=14, fontweight='bold', pad=20
)

ax.view_init(elev=25, azim=-55)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

fig.colorbar(surf, ax=ax, shrink=0.55, aspect=12, pad=0.1, label='Implied Vol (%)')
plt.tight_layout()
plt.show()

print(f"\nSurface: {vol_smooth.shape[0]} expiry pts x {vol_smooth.shape[1]} strike pts")
print(f"Strikes: {fine_strikes.min()*100:.2f}% to {fine_strikes.max()*100:.2f}%")
print(f"Expiries: {min(surface_expiries)}Y to {max(surface_expiries)}Y")

**Checkpoint:** In the 3D surface, look for:

- The **smile shape** repeating at each expiry slice -- the U-shaped cross-section along the strike axis
- The **term structure** along the expiry axis -- how the overall vol level changes with expiry
- The **skew gradient** -- the left wing (low strikes) should be consistently higher than the right wing, reflecting negative $\rho$
- The cyan dots are the actual data points (market or synthetic) that SABR was calibrated to -- the smooth surface is the interpolated SABR model

If you used FRED data, the ATM level reflects real recent swap rate volatility, but the smile shape is synthetic (based on typical SABR parameters). With gs_quant or Bloomberg VCUB, the entire surface would be market-observed.

---

## Limitations of SABR & Where to Go From Here

SABR is powerful but not perfect. Here are the key things to keep in mind:

**What SABR does well:**
- Fits the vol smile and skew accurately for a single expiry
- Provides a closed-form formula (Hagan) that's fast and differentiable
- Gives meaningful, interpretable parameters that traders can reason about
- Handles the transition between normal ($\beta = 0$) and lognormal ($\beta = 1$) dynamics

**Where SABR falls short:**
- **No mean reversion**: Real interest rates tend to mean-revert; SABR's forward dynamics don't capture this
- **Term structure inconsistency**: Parameters are calibrated slice-by-slice, so there's no guarantee of arbitrage-free consistency across expiries
- **Hagan approximation breaks down** for very low rates (near-zero or negative), very long expiries, or extreme strikes. Alternative formulas (e.g., Obloj corrections, normal SABR) help here.
- **Jumps not modeled**: SABR is a pure diffusion model -- sudden rate moves aren't captured

**Extensions for future work:**
- **Shifted SABR**: Replace $F^{\beta}$ with $(F + s)^{\beta}$ to handle negative rates
- **Time-dependent parameters**: Let $\alpha$, $\nu$, $\rho$ vary with time for better term structure fits
- **Multi-curve SABR**: Separate discounting and forwarding curves (post-2008 framework)
- **SABR with mean reversion**: Adds complexity but better captures long-dated behavior

### Reading the Parameters -- A Trader's Cheatsheet

| Parameter Change | What Happens to the Smile |
|:---|:---|
| $\alpha$ up | Entire smile shifts up (higher vol level) |
| $\nu$ up | Wings get wider (more curvature / fatter tails) |
| $\rho$ more negative | Left wing rises, right wing falls (steeper skew) |
| $\beta$ toward 1 | More lognormal behavior (vol scales with rate level) |
| $\beta$ toward 0 | More normal behavior (vol independent of rate level) |

---

*This notebook provides a complete, self-contained SABR calibration pipeline. For production use, swap in real market data from Bloomberg VCUB, add proper discounting via the swap curve, and potentially move to a shifted SABR formulation for robustness in low-rate environments.*